<a href="https://colab.research.google.com/github/SarahRinke/techlabs_trees_26/blob/moritz_dev/TechLabs_Gr2_SoSe2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datenquellen

## Baumkataster der Stadt Münster:
https://opendata.stadt-muenster.de/dataset/digitales-baumkataster-m%C3%BCnster/resource/53c9f6f3-ca4b-4418-b551-52d0e6b1b020
(csv-Datei; alternativ GEOJSON)

Datenstruktur (hat 43.115 Zeilen):

| WKN                                     | str_sch | baumgruppe |
| :-------------------------------------- | :----- | :--------- |
| POINT (7.61234661409564 51.9746341864951) | 2505 | Carpinius  |

## GEO TIFFs NRW
neben dem Geoportal NRW gibt's zum selben Datenbestand das Portal --> https://www.tim-online.nrw.de/tim-online2/
- in oberer Leiste 'Download' klicken öffnet Editor
- 1. Schritt 'Produkt': Luftbild auswählen und dann verfeinern zu Orthophotos
- 2. Datenfortmat: GEO TIFF
- 3. Ort selektieren (--> Stadt Münster hat mehrere Kacheln)
- 4. Herunterladen (jede Kachel gleich 1 Datei)

Kennzeichen:
- Ein Pixel entspricht einem Raster von 1dmx1dm
- Lage / Lageangabe: ETRS89/UTM32 (EPSG 25832)

Dann der eigenen Arbeitumgebung zugänglich machen, z.B. bei Arbeit mit Colab in Google Drive-Ordner gemeinsam mit dem Jupiter Notebook ablegen.

Über die Luftbilddienste für NRW:
https://www.bezreg-koeln.nrw.de/geobasis-nrw/produkte-und-dienste/luftbild-und-satellitenbildinformationen/aktuelle-luftbild-und

## Satellitenbilder (optional)
Aus dem Erdbeobachtungsprogramm der ESA und der EU, Corpernicus, liegen durch die sogenannten Sentinel-Missionen Erdbeobachtungsdaten vor:
https://gdz.bkg.bund.de/index.php/default/webdienste/digitale-orthophotos.html

In [ ]:
# Hier kann Code hin! (Test)

### Mounting Google Drive

For larger files or persistent storage, you can mount your Google Drive directly. This allows you to access files stored in your Drive account.

In [6]:
# mounting google drive
from google.colab import drive
#drive.mount('/content/drive')

# Now you can access files in your Drive, for example:
# df_drive = pd.read_csv('/content/drive/MyDrive/TechLabs_2026SoSe_DL/Daten/your_file.csv')
# display(df_drive.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Install geopandas if you haven't already
!pip install geopandas

import geopandas

# IMPORTANT: Make sure your Google Drive is mounted. If not, uncomment and run the cell below:
# from google.colab import drive
# drive.mount('/content/drive')

# Define the path to your GeoJSON file in Google Drive
# Replace 'your_folder/your_geojson_file.geojson' with the actual path to your file
geoj_file_path = '/content/drive/MyDrive/TechLabs_2026SoSe_DL/Daten/gruen_opendata.geojson' # Example path

# Read the GeoJSON file into a GeoDataFrame
try:
    gdf = geopandas.read_file(geoj_file_path)
    print("GeoJSON file loaded successfully!")
    display(gdf.head())
except Exception as e:
    print(f"Error loading GeoJSON file: {e}")
    print("Please ensure the path is correct and your Google Drive is mounted.")

GeoJSON file loaded successfully!


,str_schl,baumgruppe,geometry
0,02505,Tilia,POINT (7.61235 51.97463)
1,02505,Tilia,POINT (7.61254 51.97467)
2,02505,Carpinus,POINT (7.61242 51.97601)
3,02505,Carpinus,POINT (7.61241 51.9761)
4,02505,Carpinus,POINT (7.61239 51.97637)


In [ ]:
# Install rasterio for working with GeoTIFF files if you haven't already
!pip install rasterio

import rasterio
from rasterio.merge import merge
from rasterio.plot import show
import os

# IMPORTANT: Ensure your Google Drive is mounted. If not, uncomment and run the cell below:
# from google.colab import drive
# drive.mount('/content/drive')

# Define the directory where your GeoTIFF files are located
drive_path = '/content/drive/MyDrive/TechLabs_2026SoSe_DL/Daten/'

# List your GeoTIFF filenames here (replace with your actual filenames)
geotiff_filenames = [
    'dop10rgbi_32_404_5757_1_nw_2026.tif',  # Replace with actual file name
    'dop10rgbi_32_404_5758_1_nw_2026.tif',  # Replace with actual file name
    'dop10rgbi_32_405_5757_1_nw_2026.tif',  # Replace with actual file name
    'dop10rgbi_32_405_5758_1_nw_2026.tif'   # Replace with actual file name
]

# Create full paths for each GeoTIFF file
src_files_to_mosaic = []
for fname in geotiff_filenames:
    fpath = os.path.join(drive_path, fname)
    if os.path.exists(fpath):
        src_files_to_mosaic.append(rasterio.open(fpath))
    else:
        print(f"Warning: File not found at {fpath}. Please check your filenames and path.")

if not src_files_to_mosaic:
    print("No GeoTIFF files found to mosaic. Please update 'geotiff_filenames' with correct paths.")
else:
    # Mosaic the GeoTIFFs
    print(f"Attempting to mosaic {len(src_files_to_mosaic)} GeoTIFF files...")
    mosaic, out_transform = merge(src_files_to_mosaic)

    # Update the metadata for the merged file
    out_meta = src_files_to_mosaic[0].meta.copy()
    out_meta.update({
        "driver": "GTiff",
        "height": mosaic.shape[1],
        "width": mosaic.shape[2],
        "transform": out_transform
    })

    # Define the output path for the mosaicked GeoTIFF
    output_mosaic_path = os.path.join(drive_path, 'merged_geotiff_mosaic.tif')

    # Write the mosaicked GeoTIFF to a new file in your Drive
    with rasterio.open(output_mosaic_path, "w", **out_meta) as dest:
        dest.write(mosaic)

    print(f"Mosaicked GeoTIFF saved to: {output_mosaic_path}")
    print("You can now work with 'merged_geotiff_mosaic.tif' as a single raster file.")

    # Close the source files
    for src in src_files_to_mosaic:
        src.close()

# --- Explanation for GeoJSON conversion ---
print("\n--- Regarding GeoJSON conversion ---")
print("Converting GeoTIFF (raster) to GeoJSON (vector) is not a direct 'import' but requires extracting specific features.")
print("For example, if your GeoTIFF represents different land cover types, you would need to:")
print("1. Identify the pixel values that correspond to the features you want to extract.")
print("2. Use a library like `rasterio.features` to vectorize (create polygons from) these pixels.")
print("Please specify what kind of features you wish to extract from the GeoTIFF to convert it into GeoJSON.")

In [10]:
import rasterio
import pandas as pd
import numpy as np

# Assuming gdf (GeoDataFrame) is already loaded from 'gruen_opendata.geojson'
# and `output_mosaic_path` is defined from the previous GeoTIFF mosaicking step.
# output_mosaic_path = '/content/drive/MyDrive/TechLabs_2026SoSe_DL/Daten/merged_geotiff_mosaic.tif'

# Open the mosaicked GeoTIFF
try:
    with rasterio.open(output_mosaic_path) as src_mosaic:
        # Extract coordinates (x, y) from the gdf geometry
        # GeoPandas geometries are Shapely objects; for points, we can access .x and .y
        tree_coordinates = [(point.x, point.y) for point in gdf.geometry]

        # Sample pixel values from the mosaic at each tree coordinate
        # .sample() returns an iterator of arrays, each array containing the band values for a pixel
        sampled_pixel_values = []
        for val in src_mosaic.sample(tree_coordinates):
            sampled_pixel_values.append(val)

        # Convert list of arrays to a NumPy array
        sampled_pixel_values = np.array(sampled_pixel_values)

        # Create a DataFrame for the sampled pixel values and merge with 'baumgruppe'
        # Assuming your GeoTIFF has RGB bands (3 bands), plus potentially an alpha band (4 bands)
        # Adjust column names if your GeoTIFF has different bands (e.g., IR, etc.)
        if sampled_pixel_values.shape[1] >= 3:
            pixel_df = pd.DataFrame(sampled_pixel_values[:, :3], columns=['pixel_red', 'pixel_green', 'pixel_blue'])
            if sampled_pixel_values.shape[1] == 4:
                pixel_df['pixel_alpha'] = sampled_pixel_values[:, 3]
        else:
            # Handle grayscale or single-band images
            pixel_df = pd.DataFrame(sampled_pixel_values, columns=[f'pixel_band_{i+1}' for i in range(sampled_pixel_values.shape[1])])

        # Add the 'baumgruppe' as the label
        pixel_df['baumgruppe'] = gdf['baumgruppe'].reset_index(drop=True)

        print("Successfully sampled pixel values and combined with 'baumgruppe' labels.")
        print("Head of the prepared dataset:")
        display(pixel_df.head())
        print("Information about the prepared dataset:")
        pixel_df.info()

except rasterio.errors.RasterioIOError as e:
    print(f"Error opening or reading GeoTIFF file: {e}")
    print("Please ensure 'merged_geotiff_mosaic.tif' exists at the specified path and is a valid GeoTIFF.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully sampled pixel values and combined with 'baumgruppe' labels.
Head of the prepared dataset:


,pixel_red,pixel_green,pixel_blue,pixel_alpha,baumgruppe
0,0,0,0,0,Tilia
1,0,0,0,0,Tilia
2,0,0,0,0,Carpinus
3,0,0,0,0,Carpinus
4,0,0,0,0,Carpinus


Information about the prepared dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43114 entries, 0 to 43113
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   pixel_red    43114 non-null  uint8 
 1   pixel_green  43114 non-null  uint8 
 2   pixel_blue   43114 non-null  uint8 
 3   pixel_alpha  43114 non-null  uint8 
 4   baumgruppe   43114 non-null  object
dtypes: object(1), uint8(4)
memory usage: 505.4+ KB
